In [ ]:
!nvidia-smi  # check GPU usage (e.g. is there already memory blocked)

In [ ]:
import os
# NOTE!
# this is for a multi-gpu setup. we basically set which GPU is visible for the program.
# if you only have one GPU (most likely the case ;) ), make sure this is set to "0"!
# or remove this line completely!
# otherwise you may accidentally make your GPU "invisible" to the program!
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# this is important on our server, as the default /tmp directory is very small.
# so if you compile lots of models, it will eventually fill up.
# if you are working on your own machine, you might want/have to remove this!
if os.path.isdir("/var/tmp"):
    os.environ["TMPDIR"] = "/var/tmp"
    os.environ["TORCHINDUCTOR_CACHE_DIR"] = "/var/tmp"

# First Steps: Autoencoder

These notebooks will guide you through building, training and investigating the models we learn about each week in this class.
They generally consist of three or more parts:

1. Explanations about the specific model type, things to watch out for when building or training the model, etc.
2. Just reloading a model from a checkpoint, as well as the dataset, after you have trained one using the CLI.
   This requires you to set up a `yaml` config file. We will usually provide example files highlighting the parts you might want to change as `# CRITICAL`.
   There is also a lot of "garbage" in those files that you don't need to touch.
   We include a `config.md` file in the `docs` folder of the repository to help you remember which parts to change for new models or experiments.
3. (And following) Experiments, investigations, etc.

If you are going to take the LGM class, autoencoders should be familiar for you!
If you are having trouble following this notebook on a *conceptual* level, you will likely have a hard time in this class.
Struggling with the code, on the other hand, is probably to be be expected -- we are working with a new framework, and getting used to it will take some time.
This makes it all the more important for you to really spend time experimenting with it!

In [ ]:
import numpy as np
import torch
from matplotlib import pyplot as plt

from lgm.data import ImageDataModule
from lgm.models.autoencoder import AETrainingModule
from lgm.utils import interpolate
from lgm.visualization import ImagePlotter

torch.set_float32_matmul_precision('high')
torch.backends.cudnn.benchmark = True

## Part 1: Preamble

Most models we build will follow some kind of encoder/decoder structure.
As such, we will keep re-using the same abstractions over and over.
An autoencoder is a (hopefully) familiar playground to become acquainted with them.

### General Architecture Considerations
Since the focus of this class isn't really on building state-of-the art networks, but rather understand the actual underlying probabilistic models, we focus on strong baselines following simple guiding principles to reduce the burden of choice as much as possible.
Spending excessive time on fine-tuning minute architectural details will rarely be worth it.

You can use CNNs for almost everything.
- An "encoder" CNN consists of convolutions with occasional downsampling to reduce image size, increase receptive field size, and compute successively more complex and large-scale features.
- On the other hand, a "decoder" CNN starts with small images, and uses upsampling to progressively increase image size, and go from abstract representations of the whole image to high-resolution details.

For simplicity, we will generally use a "mirror" design, where the decoder is just the "opposite" of the encoder.
This reduces the number of architectural choices we have to make by half.

As mentioned in the exercise, we recorded a video tutorial going through the most important choices, which you can find here: [LINK](https://cloud.ovgu.de/s/BaDbzLBQRgwE7yY)
Highlights are:
- You have to choose the number of "levels", where each level represents a number of convolutional blocks operating at the same resolution (image size) and outputting the same number of filters.
  There is usually down-/upsampling after each level.
  As such, you can easily restrict the number of levels into a reasonable range.
  Say you are working at 32x32 images, and are using strides of 2.
  Then the first level works at 32x32 resolution, the second at 16x16, the third at 8x8, and the fourth at 4x4.
  It clearly doesn't make sense to go below this (2x2) with a convolutional kernel size of 3x3.
  Thus, four levels would be the absolute maximum for this image size.
  You could even argue for one less level, as 4x4 is already quite small, and with 3x3 kernels and `same` padding, the deepest level will actually mostly be working on padding pixels, rather than any real image information.
  So, perhaps it makes more sense to stick with three levels, going down to 8x8 at the lowest.
  For higher-resolution images, you can add more levels.
- You will usually want to increase the number of filters with deeper levels.
  A typical pattern is to double the number for each deeper level, possibly up to some limit.
  You will absolutey be fine if you stick to this pattern!
  But sometimes, it can be useful to experiment with alternatives.
  For example, it seems that having more channels at early levels can be helpful for improving generative performance, but mindlessly doubling filters would then lead to very large models.
  Then, it can pay off to diverge from this pattern.
  For example, instead of a four-level model like `[64, 128, 256, 512]` (each number being the number of filters at that level), you may go with `[128, 128, 256, 256]`.
  In general, while for simple datasets like MNIST, 32 or even fewer filters at the earliest level seems to be fine, you will usually want to go for 64 or more for more complex datasets.
- The number of *blocks per level* mainly determines network depth.
  Clearly, a model with two blocks per level will take twice as long to run as one with just one block per level.
  However, it seems like just one block is often insufficient to properly process each resolution.
  Thus, we recommend having at least two blocks per level, especially at the early levels (high resolutions).
  At deeper levels, you may get away with just one.
  More complex datasets may profit from more blocks.

All of these are set in the `yaml` config files.
Make sure you understand the basics here, as this will come up again and again!
Here is one worked example.
Say your `model.init_args` include:
- `input_shape: [1, 32, 32]`
- `latent_dim: 8`
- `style: "fc"`
- `body_config:`
    - `blocks_per_level: [2, 2, 1, 1]`
    - `level_filters: [24, 48, 96, 192]`

This means:
- Data is expected to have shape `(1, 32, 32)`, i.e. one channel and 32x32 pixels.
  You **must** set this correctly -- the model generally doesn't know the shape it will operate on, as the dataset is constructed separately.
  Thus, this information must be given to the model explicitly.
- There are four convolutional levels.
    - First level has two blocks and produces outputs of shape `(24, 32, 32)`, i.e. 24 channels.
    - Second level has two blocks and produces outputs of shape `(48, 16, 16)`.
      Why now only 16x16 pixels?
      That's because the default stride setup (in the `BodyConfig` object) is `[1, 2, 2, 2]`, i.e. stride of 1 for the first block, and 2 for all others.
      This works well because striding happens at the _start_ of the level, so the first level works at the original resolution, and all others reduce it by 2.
    - Third level has just one block and produces outputs of shape `(96, 8, 8)`.
    - Fourth level: What do you think?
- The `style: fc` results in the fourth level outputs to be flattened to a vector, and then mapped via a `Linear` layer to `latent_dim`, which is 8.
  Thus, each image is ultimately encoded to an 8-dimensional vector!
- The decoder simply reverts everything, and "decodes" the 8-dimensional latent vector back to a `(1, 32, 32)` image.

### Autoencoder-specific: Latent Space Size
Above, we used a `latent_dim` of 8.
Clearly, having more latent dimensions makes the autoencoding task easier, as it provides more "space" to put information in.
We recommend that you experiment with this value, and train models with different sizes.
In general, you should see improvements in reconstruction loss, as well as visual quality, as you increase `latent_dim`.

### Autoencoder-specific: Data Likelihood
As we discussed in the exercise, we have to choose a probability distribution that we believe the data was sampled from.
The network then has to output the _parameters_ of the distribution.
This is crucial:
In a probabilistic model like this, the network outputs are _not_ themselves the data samples/reconstructions (although there are "implicit" generative models where this _is_ the case!);
they are _distribution parameters_.
We can then optimize the network weights for maximum likelihood, i.e. such that the data gets the maximum probability under the assumption that it comes from the distribution we chose.
In order to get proper reconstructions, we have to _transform_ the distribution parameters, in some way;
for example, we may output the _expected value_ (i.e. the mean) of the distribution, and display that as the actual reconstructed/sampled images.

Choosing a distribution is a complex topic.
We have implemented various likelihoods in the repository, with corresponding loss functions and expected value transformations.
Probably the simplest choice is `GaussianFixedSigma` -- a Normal distribution with fixed variance, which then completely drops out of the loss and expected value.
Thus, the model only has to estimate the mean $\mu$.
This leads to the squared difference loss function, and since the model outputs the Normal distribution parameter $\mu$, and the expected value is itself just $\mu$, the network outputs can actually be taken as reconstructions directly!

Let's consider a more complex example as a contrast.
Say we are assuming a Beta distribution ([LINK](https://en.wikipedia.org/wiki/Beta_distribution)) instead.
This distribution has two parameters $\alpha$ and $\beta$, and it's defined for data in the interval (0, 1), so it makes sense for image data with pixels normalized into that range.
Other than for the Gaussian, we can't really ignore either parameter -- we need both to properly set the location and form of the distribution.
This means our network has two output _two values_ for each pixel in order to get a proper Beta distribution!
This clearly shows that you can't just plot the network outputs as reconstructions or samples -- there are more outputs than there were pixels in the input!
Instead, you could use the expected value, which is just $\frac{\alpha}{\alpha + \beta}$.
Try it out:
Use `lgm.likelihoods.Beta` in the config instead, and train a model with it.
Issues such as the decoder having to output twice as many values, or mapping them to the expected value, are all handled by our code base!

### Autoencoder-specific: Dequantizing Data
Images are usually stored as unsigned 8-bit integers, meaning numbers between 0 and 255.
Thus, there are only 256 unique values.
We usually convert to floats and normalize to range [0, 1], but that clearly doesn't change the fact that there are still only 256 unique pixel values in that range.
Most of the space will be completely empty!

However, by applying likelihoods such as the Gaussian or Beta, we are assuming _continuous_ data.
Continuous probability distributions have the property that, given enough flexibility, they can "pile on" an essentially infinite amount of probability mass onto an infinitely narrow range.
By passing 8-bit data, which only inhabits small "pockets" in the continuous space, we give our models the opportunity to do exactly that.
This can lead to the model "cheating" by identifying parts of the data that are easy to model (e.g. background pixels in MNIST), and just become more and more certain about those specific values.
This, in turn, can lead to likelihoods becoming arbitrarily large, even though the model might not even be doing that well, according to visual judgement.

The solution is to "dequantize" the data, which means filling out the empty space between the "8-bit pockets" of data.
The easiest way to do this is by adding random noise, most commonly uniform.
This is why our config includes `UniformDequantize` as an `additional_transform`.
This transform is crucial for likelihood-based models, although technically speaking, this depends on the likelihood.
For example, `GaussianFixedSigma` is actually "safe":
Since $\sigma$ is fixed and can't be reduced, this likelihood lacks the flexibility to collapse onto a single point.

### General Training Considerations
You don't have to do much for the general training process.
You should set `max_epochs` in the config.
Generative models often need some time for training.
You might try around 100 epochs or so for a dataset like MNIST, and more for more complex datasets.
Some of the models we learn about later in the class are notoriously slow to train, with literature often recommending 100,000s of steps, which might be something like a 1000 epochs or more, depending on batch size.

Aside from the number of steps, you can twiddle with the `optimization_config`, setting values such as learning rate, weight decay, or warmup steps.
You can also write new configs for other optimizers if you want -- see `lgm.optimizers.AdamWConfig` for an example that you could replicate for different optimizers, or `CosineWarmupConfig` for a learning rate scheduler example.

### Autoencoder-specific: Reconstructions
The only metric of interest for now is the reconstruction error.
Be sure to set the correct metric to `monitor` in the config for the `ModelCheckpoint` callback.
We also include a `ReconstructionLogging` callback which is responsible for regularly creating reconstructions for some samples from the test set, and plotting them to Tensorboard.
This is a great way to visually judge the training progress.
Do you feel like the changes in reconstruction loss match the visual improvements of the reconstructions?

## Part 2: Reloading

### Model
Super simple! We literally just have to "load from checkpoint", and all the information on how to construct the architecture etc. is contained therein.
The model summaries (printed below) tend to be rather verbose, as we have many nested levels of different stuff.

Be sure to set `dataset`, `name`, and `version` to the correct values for whatever you want to load, and adapt `check_path` in case you decided for a different folder structure.

In [ ]:
dataset = "mnist"
name = "baseline"
version = 0

check_path = f"lightning_logs/autoencoder/{dataset}/{name}/version_{version}/checkpoints/last.ckpt"

lightning_module = AETrainingModule.load_from_checkpoint(check_path)
model = lightning_module.model.eval()
model.compile()
device = lightning_module.device
print("Device is", device)

In [ ]:
print(model)

### Data
We can even load the data module from the same checkpoint!
It would also be possible to override things like `batch_size` or `num_workers` if you want -- they really wouldn't need to be the same as what you used for training.
If you want more control, you could just do something like `data = ImageDataModule("mnist", batch_size=128, num_workers=12)`.
This way, you could even do silly things like load a different dataset than the autoencoder was actually trained for, and see how well it works there.
But be mindful that you would need to specify transforms etc. yourself.

Note: At this time, our data module ignores the `stage` argument in the `setup` function, so it doesn't really matter what we put in there.

In [ ]:
data_module = ImageDataModule.load_from_checkpoint(check_path)
data_module.prepare_data()
data_module.setup()

The `plotter` object below is needed for part 3, but we may as well create it now to make sure it's available.

In [ ]:
plotter = ImagePlotter(transform=data_module.normalizer.invert)

## Part 3: Exploration

This is where we start looking at new concepts!

### Generating Images
**Note:** This tends to work much better with a smaller `latent_dim` for the model, say, something around 8 rather than something like 128 or more.
However, such small models won't scale well to more complex datasets.
In fact, even on MNIST, reconstructions will be significantly worse (very blurry) for a very small latent space.

Our decoder maps from a lower-dimensional "code space" to images.
Can we somehow use this to generate new images that look like our training data?
The hope here is that the code space has a simpler structure that makes it easy to generate new codes, which can then be mapped to images (which have a much more complex distribution).
As such, we could try to come up with a distribution for the code space.
Arguably the simplest idea is to check what range the code values are in (this can differ per dimension) and then generate uniformly from that range.
In order to do that, we have to encode the entire dataset first.

In [ ]:
all_codes = []
with torch.inference_mode():
    for input_batch, _ in data_module.train_dataloader():
        batch_encoded = model.encoder(input_batch.to(device))
        all_codes.append(batch_encoded)
all_codes = torch.cat(all_codes, dim=0)

To keep things compact, the below function also already implements the methods we will try further below, but for now you can just look at the `if method == "uniform"` block.

In [ ]:
def generate_images(method: str):
    if method == "uniform":
        code_mins = all_codes.amin(dim=0)
        code_maxs = all_codes.amax(dim=0)
        distribution = torch.distributions.Uniform(code_mins, code_maxs)
    elif method == "gaussian_diagonal":
        code_means = all_codes.mean(dim=0)
        code_stddevs = torch.std(all_codes, dim=0)
        distribution = torch.distributions.Normal(code_means, code_stddevs)
    elif method == "gaussian_full":
        code_means = all_codes.mean(dim=0)
        covariance = torch.cov(all_codes.T)
        distribution = torch.distributions.MultivariateNormal(loc=code_means, covariance_matrix=covariance)
    else:
        raise ValueError(f"Invalid method {method}")

    n_rows = 8
    random_codes = distribution.sample((n_rows**2,)).to(device)    
    with torch.inference_mode():
        generated = model.reconstruct_from_code(random_codes)

    plotter.image_grid(generated,
                       title=f"Generations using {method} method",
                       n_rows=n_rows)

In [ ]:
generate_images("uniform")

As we can see above, the results look vaguely MNIST-y -- black shapes on white backbround.
But this is clearly not what the training data looks like.
Our uniform assumption must be wrong!
We can check the actual distribution of values per code dimension.
It seems like a Gaussian may be more appropriate.

NOTE the code below has a 2-row subplot grid hardcoded.
This is intended for 8-dimensional latent space.
If you change the latent space size for your model, you may want to add more rows and change the figure size!

In [ ]:
n_rows = 2
n_cols = all_codes.shape[1] // n_rows

plt.figure(figsize=(12, 4))
for ind, codes_dim in enumerate(all_codes.T):
    plt.subplot(n_rows, n_cols, ind+1)
    plt.hist(codes_dim.cpu().numpy(), bins=100)
plt.tight_layout()
plt.show()

Thus, we can compute the _mean and standard deviation_ per dimension and generate the codes from that distribution instead:

In [ ]:
generate_images("gaussian_diagonal")

Looks better, but still far from good. It seems like our Gaussian assumption is still off.
In particular, our attempt above generates each code dimension _independently_.
However, the code dimensions learned by the model may well be correlated. We can once again check this for the dataset:

In [ ]:
covariance = torch.cov(all_codes.T)
absmax = covariance.abs().max().item()
plt.imshow(covariance.cpu().numpy(), vmin=-absmax, vmax=absmax, cmap="RdBu_r")
plt.colorbar()
plt.title("Covariance")
plt.show()

correlation = torch.corrcoef(all_codes.T)
plt.imshow(correlation.cpu().numpy(), vmin=-1, vmax=1, cmap="RdBu_r")
plt.colorbar()
plt.title("Correlation")
plt.show()

Indeed, there seem to be significant correlations between many of the code dimensions.
As such, generating each dimension independently from the others is not appropriate.
We can instead use a Gaussian distribution with a full covariance matrix:

In [ ]:
generate_images("gaussian_full")

Looks better!
And yet, still not like the MNIST dataset.
There are many "non-digits" as well as blurry images.
Looking at the distributions above, the Gaussian assumption is clearly just an approximation;
the actual distributions look more complex.
There may also be non-linear relations between code dimensions, whereas the covariance matrix only models linear ones.

This course will deal with how to build true generative models in a principled fashion.
These models are built from the ground up to be able to generate new samples that look like they are from the training dataset.
Even with this simple example, there are already several open questions:
- What is the "correct" loss function?
- How do we compare different losses, models or generation functions in a principled manner?
  That is, how do we judge the quality of generations?
- Can we encourage encodings to follow a certain distributions?

### Latent Space Exploration
Finally, we can also look at _interpolating_ two images in the latent space.
We encode both images, then interpolate between the codes, decoding the intermediate codes at various points.
The results should smoothly transform one image to the other, although this sometimes fails, especially on simple MNIST-like datasets.

There are many other things we can do with latent representations!
But this should suffice for now.

In [ ]:
# pick two random images and plot
randind1 = np.random.randint(len(data_module.test_data))
randind2 = np.random.randint(len(data_module.test_data))

print(f"Picked indices {randind1}, {randind2}")
# 3914, 4355

# first get random index, then 0 to only get the image (not label)
image1 = data_module.test_data[randind1][0]
image2 = data_module.test_data[randind2][0]
images = torch.stack((image1, image2))

with torch.inference_mode():
    codes = model.encoder(images.to(device))
code1, code2 = codes

# see the images we are dealing with here
# NOTE plotter itself plots the images, and calling the expression with no return in a notebook will
# result in a second (i.e. duplicated) plot. I haven't found a way to fix this yet, which is why
# there is a dummy "_" return; this suppresses the second plot.
_ = plotter.image_grid(images,
                       title="",
                       n_rows=1,
                       n_cols=2)

NOTE that the 2D grid layout implies there's something 2D happening here.
It's not.
It just saves space to display the images this way.
It's a 1d interpolation from top left to bottom right.
It would be clearer to plot it all in one row or column.

In [ ]:
n_rows = 8
interpolated_codes = interpolate(code1, code2, n_interpolations=n_rows**2)

with torch.inference_mode():
    interpolated_images = model.reconstruct_from_code(interpolated_codes)
_ = plotter.image_grid(interpolated_images,
                       title="Latent Interpolations",
                       n_rows=n_rows, suppress_plots=True)

Compare this to interpolating directly in the image space:
One image just slowly disappears, while the other appears in the background like a "ghost".

In [ ]:
interpolated_images_naive = interpolate(image1, image2, n_interpolations=n_rows**2)
_ = plotter.image_grid(interpolated_images_naive,
                       title="Naive Interpolations",
                       n_rows=n_rows)

We even have code to make a little animation!
We concatenate the series of images with itself in reverse (removing endpoints, which would otherwise be duplicated) to get a smooth "back-and-forth" morph.

In [ ]:
plotter.make_animation(f"animations/interpolate_ae_{dataset}",
                        torch.cat((interpolated_images, 
                                   torch.flip(interpolated_images[1:-1], dims=[0]))),
                        invert_colors=True)